# Ear Biometrics — Phase 4: AMI SweepReproduction + extension of Mohamed et al., arXiv:2406.00135.**Before running:** attach both datasets via *Add Input* (right panel) —`ami-ear-database` and `EarVN-database` — and set Accelerator to **GPU P100**under Settings. The paper used a Kaggle P100, so this makes our compute numbersdirectly comparable to theirs.Run cells top to bottom. Cell 6 is the long one.

## 1. Check the environment and find the data

In [ ]:
import subprocess, sys, osfrom pathlib import Path# --- GPU check: do this FIRST, before wasting time on anything else ---try:    import torch    if torch.cuda.is_available():        print(f"GPU : {torch.cuda.get_device_name(0)}")        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")    else:        print("!! NO GPU. Settings -> Accelerator -> GPU P100, then restart the session.")except ImportError:    print("torch not found (unexpected on Kaggle)")# --- Locate the datasets by SEARCHING, not guessing ---# Kaggle's unzip layout depends on how the zip was made, so we find the folder# that actually contains the images instead of hardcoding a path.def find_ami(root="/kaggle/input"):    for p in Path(root).rglob("*_back_ear.jpg"):        return p.parent    return Nonedef find_earvn(root="/kaggle/input"):    for p in Path(root).rglob("*.ALI_HD"):        return p.parent          # the folder holding the subject folders    return NoneAMI_DIR   = find_ami()EARVN_DIR = find_earvn()print(f"\nAMI   : {AMI_DIR}")if AMI_DIR:   print(f"        {len(list(AMI_DIR.glob('*.jpg'))):,} jpg  (expect 700)")print(f"EarVN : {EARVN_DIR}")if EARVN_DIR:    subs = [d for d in EARVN_DIR.iterdir() if d.is_dir()]    print(f"        {len(subs)} subject folders  (expect 164)")assert AMI_DIR and EARVN_DIR, "One or both datasets not found -- check Add Input"

## 2. Clone the code

In [ ]:
%cd /kaggle/working!rm -rf ear-biometrics!git clone --quiet https://github.com/Zwelllll/ear-biometrics.git%cd /kaggle/working/ear-biometrics!git log --oneline -1

## 3. Point the config at the real Kaggle pathsOverwrites `datasets.*.kaggle` with the paths discovered in cell 1, so thenotebook keeps working even if you re-upload a dataset with different nesting.

In [ ]:
import yaml, syssys.path.insert(0, "/kaggle/working/ear-biometrics")cfg_path = Path("/kaggle/working/ear-biometrics/config.yaml")cfg = yaml.safe_load(cfg_path.read_text())cfg["datasets"]["ami"]["kaggle"]   = str(AMI_DIR)cfg["datasets"]["earvn"]["kaggle"] = str(EARVN_DIR)cfg["data"]["num_workers"] = 2      # Kaggle gives ~4 vCPU; 2 is the sweet spotcfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))from src.config import CFGfrom src.paths import Pprint(f"env   : {P.env}")print(f"cache : {P.data}/cache        <- must be writable (/kaggle/temp)")print(f"ami   : {CFG.datasets['ami']['kaggle']}")print(f"earvn : {CFG.datasets['earvn']['kaggle']}")print(f"canny : ami {CFG.preprocessing.canny['thresholds']['ami']}  "      f"earvn {CFG.preprocessing.canny['thresholds']['earvn']}")

## 4. Manifests and splitsThese must reproduce the local numbers exactly:**AMI 700 / 100 subjects**, **EarVN 28,412 / 164 subjects**.A mismatch means the upload lost files. The leakage assertions run automatically.

In [ ]:
!python -m src.manifest --dataset all!python -m src.splits --dataset all

## 5. Preprocessing cacheWrites `raw`, `zoom` and `zoom_canny` at 224x224 into `/kaggle/temp/cache`.~87k files total. Kaggle's disk is fast, so expect a few minutes.

In [ ]:
!python -m src.preprocess --dataset ami!python -m src.preprocess --dataset earvn

## 5b. Look at the contact sheetsDon't skip this. It's the cheapest bug-catcher available: a broken crop or ablank edge map is obvious here and invisible in an accuracy number.

In [ ]:
from IPython.display import Image, displayfor ds in ("ami", "earvn"):    p = Path(f"/kaggle/working/ear-biometrics/results/contact_sheet_{ds}.png")    if p.exists():        print(f"=== {ds} ===")        display(Image(str(p)))

## 6. The AMI sweep — 45 runs3 models x 5 arms x 3 seeds. AMI has 500 training images, so each epoch isseconds on a P100.**If the session dies partway**, just re-run this cell: `sweep.py` skips everyrun already recorded in `results.csv` and continues where it stopped.

In [ ]:
# Sanity check first -- one batch through the biggest model, ~10 seconds.!python -m src.train --dataset ami --model resnet50 --arm zoom+aug --seed 0 --dry-run

In [ ]:
# See what is queued before committing GPU time to it.!python -m src.sweep --dataset ami --dry

In [ ]:
# The real thing.!python -m src.sweep --dataset ami

## 7. Results`results.csv` and the best checkpoints are copied into `/kaggle/working` so theysurvive the session. **Download `results.csv` and commit it to your repo** —that file is your results table.

In [ ]:
import shutil, pandas as pdfrom src.results import summarisesrc_csv = Path("/kaggle/working/ear-biometrics/results/results.csv")if src_csv.exists():    shutil.copy(src_csv, "/kaggle/working/results.csv")    df = pd.read_csv(src_csv)    print(f"{len(df)} runs recorded\n")    # mean +/- std across seeds -- the thing the paper never reported    s = summarise(group_cols=("train_dataset", "model", "arm"), metric="test_top1")    s["mean"] = (s["mean"] * 100).round(2)    s["std"]  = (s["std"] * 100).round(2)    print(s.to_string(index=False))else:    print("no results yet")

In [ ]:
# Pivot: arms as columns, models as rows -- report-table shape.if src_csv.exists():    d = pd.read_csv(src_csv)    d = d[d["test_top1"].notna() & (d["train_dataset"] == "ami")]    piv = d.pivot_table(index="model", columns="arm", values="test_top1",                        aggfunc=["mean", "std"])    print((piv * 100).round(2).to_string())

In [ ]:
# Keep the best checkpoints for Phase 6 (cross-dataset) and Phase 7 (verification).ck = Path("/kaggle/working/ear-biometrics/checkpoints")best = sorted(ck.glob("*__best.pth"))print(f"{len(best)} best-checkpoints, "      f"{sum(f.stat().st_size for f in best)/1e9:.2f} GB total")out = Path("/kaggle/working/checkpoints"); out.mkdir(exist_ok=True)for f in best:    shutil.copy(f, out / f.name)print(f"copied to {out}")

## Next**Phase 5** — the EarVN sweep. Far more expensive (28,412 images vs 700), so runa single seed first:```!python -m src.sweep --dataset earvn --seeds 0```then add seeds 1 and 2 only for the specific rows you'll make claims about.